In [4]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath('..'))

fdi_csv = Path('../data/fdi_Detection/Big_Datasets_for_Intelligent_Anomaly_Detection_Mi/2(c) current and voltage_timeseries_big_all_with_anomalies.csv')
print(f"Loading FDI Data from: {fdi_csv.name}")
df_fdi = pd.read_csv(fdi_csv)

# Composite attack label: 1 if ANY sensor channel is manipulated
df_fdi['is_attack'] = (
    df_fdi['is_anomaly_voltage'] | 
    df_fdi['is_anomaly_current'] | 
    df_fdi['is_anomaly_frequency']
).astype(int)

print("--- Telemetry & Attack Summary ---")
print(f"Total raw rows: {len(df_fdi)}")
print(f"Anomaly breakdown:\n{df_fdi[['is_anomaly_voltage', 'is_anomaly_current', 'is_anomaly_frequency', 'is_attack']].sum()}")

# Pivot to multivariate matrix: (Timestamps x [4 Feeders * 3 Metrics = 12 Features])
features_pivoted = df_fdi.pivot_table(
    index='time_s',
    columns='feeder',
    values=['voltage_kV', 'current_A', 'frequency_Hz']
)
features_pivoted.columns = [f"{col[0]}_{col[1]}" for col in features_pivoted.columns]

# Ground truth label per time step: 1 if any feeder at that second had an active attack
labels_pivoted = df_fdi.groupby('time_s')['is_attack'].max()

print("\n--- Synchronized Matrix ---")
print(f"Features Shape: {features_pivoted.shape}")
print(f"Labels Shape:   {labels_pivoted.shape}")
print(f"Attacked Steps: {labels_pivoted.sum()} / {len(labels_pivoted)} ({labels_pivoted.mean()*100:.2f}%)")
features_pivoted.head(2)

Loading FDI Data from: 2(c) current and voltage_timeseries_big_all_with_anomalies.csv
--- Telemetry & Attack Summary ---
Total raw rows: 28804
Anomaly breakdown:
is_anomaly_voltage      24
is_anomaly_current      32
is_anomaly_frequency    20
is_attack               75
dtype: int64

--- Synchronized Matrix ---
Features Shape: (7201, 12)
Labels Shape:   (7201,)
Attacked Steps: 75 / 7201 (1.04%)


,current_A_12 kV Feeder,current_A_3 kV Feeder,current_A_5 kV Feeder,current_A_8 kV Feeder,frequency_Hz_12 kV Feeder,frequency_Hz_3 kV Feeder,frequency_Hz_5 kV Feeder,frequency_Hz_8 kV Feeder,voltage_kV_12 kV Feeder,voltage_kV_3 kV Feeder,voltage_kV_5 kV Feeder,voltage_kV_8 kV Feeder
time_s,,,,,,,,,,,,
0.0,65.432395,121.872511,108.474117,83.927950,50.016963,50.030449,50.047722,50.027751,12.045215,3.006911,5.040968,8.043861
0.5,66.105080,123.635813,108.330603,85.066468,50.016909,50.041327,50.049694,50.030485,12.032136,3.017087,5.035435,8.051990


In [5]:
from sklearn.preprocessing import MinMaxScaler

# 1. Chronological Train / Val / Test Split
n = len(features_pivoted)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train_raw = features_pivoted.iloc[:train_end].values
y_train_raw = labels_pivoted.iloc[:train_end].values

X_val_raw = features_pivoted.iloc[train_end:val_end].values
y_val_raw = labels_pivoted.iloc[train_end:val_end].values

X_test_raw = features_pivoted.iloc[val_end:].values
y_test_raw = labels_pivoted.iloc[val_end:].values

# 2. Fit MinMaxScaler strictly on the training partition
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled = scaler.transform(X_val_raw)
X_test_scaled = scaler.transform(X_test_raw)

# 3. Create Sliding Windows
# window_size = 30 steps (15 seconds of dynamic telemetry at 0.5s resolution)
WINDOW_SIZE = 30

def create_labeled_windows(features: np.ndarray, labels: np.ndarray, window_size: int = 30):
    num_samples = len(features) - window_size + 1
    X_wins, y_wins = [], []
    for i in range(num_samples):
        X_wins.append(features[i : i + window_size])
        # Window label is 1 if any attack occurs within that temporal span
        y_wins.append(1.0 if np.any(labels[i : i + window_size] == 1) else 0.0)
    return np.array(X_wins, dtype=np.float32), np.array(y_wins, dtype=np.float32)

X_train_win, y_train_win = create_labeled_windows(X_train_scaled, y_train_raw, WINDOW_SIZE)
X_val_win, y_val_win = create_labeled_windows(X_val_scaled, y_val_raw, WINDOW_SIZE)
X_test_win, y_test_win = create_labeled_windows(X_test_scaled, y_test_raw, WINDOW_SIZE)

print("--- Sliding Window Datasets Ready ---")
print(f"X_train: {X_train_win.shape}, Attacked Windows: {int(y_train_win.sum())} / {len(y_train_win)}")
print(f"X_val:   {X_val_win.shape}, Attacked Windows: {int(y_val_win.sum())} / {len(y_val_win)}")
print(f"X_test:  {X_test_win.shape}, Attacked Windows: {int(y_test_win.sum())} / {len(y_test_win)}")

--- Sliding Window Datasets Ready ---
X_train: (5011, 30, 12), Attacked Windows: 1581 / 5011
X_val:   (1051, 30, 12), Attacked Windows: 262 / 1051
X_test:  (1052, 30, 12), Attacked Windows: 210 / 1052


In [6]:
import sys
import os
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

sys.path.append(os.path.abspath('..'))
from src.evaluation import evaluate_fdi_model
from src.models import FDIDetectorBaseline

# Verify device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training Baseline Model on: {device}')

# DataLoaders
train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train_win), torch.from_numpy(y_train_win)),
    batch_size=64,
    shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_val_win), torch.from_numpy(y_val_win)),
    batch_size=64,
    shuffle=False,
)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_test_win), torch.from_numpy(y_test_win)),
    batch_size=64,
    shuffle=False,
)

# Model setup
num_features = X_train_win.shape[2]
model = FDIDetectorBaseline(
    in_features=num_features, cnn_dim=64, lstm_hidden=64
).to(device)

# Handle class imbalance
num_pos = y_train_win.sum()
num_neg = len(y_train_win) - num_pos
pos_weight = torch.tensor([num_neg / (num_pos + 1e-5)]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Training loop
EPOCHS = 15
print(f'\n--- Training Scratch Baseline for {EPOCHS} Epochs ---')
for epoch in range(1, EPOCHS + 1):
  model.train()
  running_loss = 0.0
  for x_b, y_b in train_loader:
    x_b, y_b = x_b.to(device), y_b.to(device)
    optimizer.zero_grad()
    logits = model(x_b)
    loss = criterion(logits, y_b)
    loss.backward()
    optimizer.step()
    running_loss += loss.item() * len(y_b)

  epoch_loss = running_loss / len(train_loader.dataset)
  if epoch % 3 == 0 or epoch == 1:
    print(f'Epoch [{epoch:02d}/{EPOCHS:02d}] - Loss: {epoch_loss:.4f}')

# Save weights
save_model_path = Path('../saved_models')
save_model_path.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), save_model_path / 'scratch_baseline_fdi.pth')
print(f'\nModel weights saved to: {save_model_path / "scratch_baseline_fdi.pth"}')

# Evaluate
metrics = evaluate_fdi_model(
    model=model,
    test_loader=test_loader,
    device=device,
    save_dir='../results',
    model_name='scratch_baseline',
)

Training Baseline Model on: cuda

--- Training Scratch Baseline for 15 Epochs ---
Epoch [01/15] - Loss: 0.8971
Epoch [03/15] - Loss: 0.3518
Epoch [06/15] - Loss: 0.1766
Epoch [09/15] - Loss: 0.1317
Epoch [12/15] - Loss: 0.1051
Epoch [15/15] - Loss: 0.0794

Model weights saved to: ..\saved_models\scratch_baseline_fdi.pth

=== Evaluation Report: scratch_baseline ===
Accuracy:  0.8394
Precision: 0.5981
Recall:    0.5952
F1-Score:  0.5967
ROC-AUC:   0.8440
Confusion Matrix:
[[758  84]
 [ 85 125]]
